# DocVQA Zero-Shot Baseline - Google Colab (Production Ready)

Complete setup and evaluation pipeline for the VisionDocPhi-3.5 project.

This notebook:
- ✅ Sets up the production-ready project structure
- ✅ Installs all dependencies
- ✅ Loads Phi-3.5 Vision model
- ✅ Runs zero-shot VQA on DocVQA dataset
- ✅ Calculates metrics (ANLS, Exact Match)
- ✅ Saves results

## Step 1: Clone from GitHub & Setup Project

In [ ]:
import os
import sys

# ============ STEP 1: Clone GitHub Repository ============
PROJECT_NAME = "VisionDocPhi-3.5"
GITHUB_REPO = "https://github.com/mokshu7k/VisionDocPhi-3.5.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

# Clone repository
if not os.path.exists(PROJECT_PATH):
    print("📥 Cloning repository from GitHub...")
    !git clone {GITHUB_REPO} {PROJECT_PATH}
    print("✅ Repository cloned successfully!\n")
else:
    print(f"✓ Repository already exists at {PROJECT_PATH}\n")

# Change to project directory
os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

print(f"📂 Working directory: {os.getcwd()}\n")

## Step 2: Pull Latest Changes (Optional)

In [ ]:
# Pull latest changes from GitHub (run this if you made updates to the repo)
# !git pull origin main
print("✓ Ready to pull updates if needed: git pull origin main")

## Step 3: Install Dependencies

In [ ]:
import os
import sys

# Ensure we're in the correct directory
os.chdir('/content/VisionDocPhi-3.5')

# ⚠️ COLAB NOTE: PyTorch is pre-installed. We only upgrade specific packages.
print("📦 Installing dependencies...\n")

# Fix Pillow compatibility issue first (use version <12.0 for gradio compatibility)
print("🔧 Installing compatible Pillow version...")
!pip install -q Pillow==11.1.0

# First, upgrade pip and core dependencies (avoid downgrading PyTorch)
print("📥 Upgrading core dependencies...")
!pip install -q --upgrade transformers huggingface-hub scipy scikit-learn tqdm

# Install ML-specific packages
print("📦 Installing ML packages...")
!pip install -q gradio diffusers peft datasets accelerate

# Install bitsandbytes for 8-bit quantization
print("\n🔷 Installing bitsandbytes for 8-bit quantization...")
!pip install -q bitsandbytes 2>/dev/null || echo "⚠️  Note: bitsandbytes installation may need CUDA build tools"

# Optional: Install flash-attn for faster GPU inference
print("\n⚡ Installing FlashAttention2 for GPU optimization...")
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "⚠️  FlashAttention2 not available (will use eager attention)"

print("\n✅ All dependencies installed successfully!")
print("Note: PyTorch was kept at Colab's pre-installed version for CUDA compatibility")


✅ Dependencies installed!


In [ ]:
# ============ GPU MEMORY OPTIMIZATION ============
import os
import torch

# Memory optimizations configured:
# - 8-bit Quantization: Enabled (via bitsandbytes) - Reduces memory by ~75%
# - Gradient Checkpointing: Enabled (saves memory during inference)
# - Eager Attention: Enabled for better Colab compatibility
# - Memory Cleanup: Automatic every 5 batches
# - Low CPU Memory Usage: Enabled during model loading

print("✅ Memory optimizations configured:")
print("  🔷 8-bit Quantization: Enabled (bitsandbytes)")
print("  📦 Gradient Checkpointing: Enabled")
print("  ⚡ Attention Mode: Eager (optimized for Colab)")
print("  🧹 Memory Cleanup: Every 5 batches")
print("  💾 Model Loading: Low CPU memory mode")

# Check GPU memory available
if torch.cuda.is_available():
    print(f"\n💾 GPU Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"📊 Initial GPU Memory Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("\n⚠️  CUDA not available - will use CPU (much slower!)")


In [ ]:
from google.colab import files
import os

os.chdir('/content/VisionDocPhi-3.5/data/raw')

# Replace with YOUR FILE ID from Google Drive
FILE_ID = "1hYoCymrrY9i106Y0RSPZB8P8l9AS9Tq4"

print(f"📥 Downloading images from Google Drive...")
!pip install -q gdown
!gdown {FILE_ID} -O spdocvqa_images.zip

print("\n🔓 Extracting zip file...")
!unzip -q spdocvqa_images.zip

print("\n✅ Done! Images are ready!")

# Verify
images_dir = '/content/VisionDocPhi-3.5/data/raw/spdocvqa_images'
num_images = len(os.listdir(images_dir))
print(f"✓ Found {num_images} images")

In [ ]:
# ============ CHUNKED EVALUATION (FULL DATASET) ============
# Recommended for processing all images with limited GPU memory
# Processes in chunks, saves progress, resumes automatically

import os
os.chdir('/content/VisionDocPhi-3.5')

print("🚀 Starting Chunked Evaluation on Full Dataset\n")
print("This will:")
print("  ✅ Process the full validation set in 200-sample chunks")
print("  ✅ Save progress after each chunk")
print("  ✅ Automatically resume if interrupted")
print("  ✅ Merge results from all chunks\n")

# Run chunked evaluation
!python scripts/chunked_evaluation.py --split val --chunk_size 200 --resume

print("\n✅ Chunked evaluation complete!")
print("📊 Results saved to: data/outputs/")


## Step 5: Chunked Evaluation (Recommended for Full Dataset)

For processing the **entire validation/test set** without GPU memory errors:
- Automatically processes in 200-sample chunks
- Saves progress after each chunk
- Can resume if interrupted
- Automatically merges all results

Use this instead of the quick test above to get results on the full dataset.


## Step 3.5: Download Dataset (Optional)

⚠️ **Only required if running full evaluation (Steps 10+)**

Skip this if you only want to test the model without the dataset.

**Note on Quantization:** The model uses 8-bit quantization via bitsandbytes (enabled by default), which reduces memory usage by ~75%. This makes it feasible to run even on Colab's T4 GPU.


## Step 4: Verify Project Setup

In [ ]:
import os
import sys
from pathlib import Path
import torch

print("\n" + "="*70)
print("🔍 VERIFICATION CHECKLIST")
print("="*70 + "\n")

# Check directory structure
print("📁 Checking project structure...")
required_dirs = ['config', 'src', 'scripts', 'notebooks', 'data/raw', 'data/outputs']

for dir_name in required_dirs:
    if os.path.exists(dir_name):
        print(f"  ✓ {dir_name}/")
    else:
        print(f"  ✗ {dir_name}/ NOT FOUND")

print("\n📦 Checking PyTorch & CUDA...")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

print("\n🔷 Checking Quantization Requirements...")
try:
    import bitsandbytes
    print(f"  ✓ bitsandbytes: {bitsandbytes.__version__}")
except ImportError:
    print(f"  ✗ bitsandbytes NOT FOUND - Install with: pip install bitsandbytes")

print("\n📊 Checking dataset files...")
if os.path.exists('data/raw/spdocvqa_qas/val_v1.0_withQT.json'):
    print("  ✓ Validation annotations found")
else:
    print("  ✗ Validation annotations NOT found")
    print("    Download from: https://rrc.cvc.uab.es/?ch=17")

if os.path.exists('data/raw/spdocvqa_images'):
    num_images = len(os.listdir('data/raw/spdocvqa_images'))
    print(f"  ✓ Images directory ({num_images} items)")
else:
    print("  ✗ Images directory NOT found")

print("\n" + "="*70)



🔍 VERIFICATION CHECKLIST

📁 Checking project structure...
  ✗ config/ NOT FOUND
  ✗ src/ NOT FOUND
  ✗ scripts/ NOT FOUND
  ✗ notebooks/ NOT FOUND
  ✗ data/raw/ NOT FOUND
  ✗ data/outputs/ NOT FOUND

📦 Checking PyTorch...
  PyTorch: 2.12.0+cpu
  CUDA available: False
  Device: cpu

📊 Checking dataset files...
  ✗ Validation annotations NOT found
    Download from: https://rrc.cvc.uab.es/?ch=17
  ✗ Images directory NOT found



## Step 5: Import Project Modules

### ⚠️ IMPORTANT: Restart Runtime (Before Step 5)

**Do NOT reload the page.** Just restart the runtime within Colab:

1. Click the **Runtime** menu at the top
2. Click **Restart runtime** (or **Restart session**)
3. Wait 30 seconds for it to fully restart
4. Then run Step 5 (Import Project Modules) below

This clears all cached Python modules and ensures fresh imports work correctly.

**If you get PIL/Pillow errors:**
- Go back to Step 3 and run it again
- Then restart runtime again
- Then proceed with Step 5

**If you can't find "Restart runtime":**
- Look for: Runtime → Restart session
- OR: Runtime → Disconnect and delete runtime (then refresh page once)

In [ ]:
import os
import sys
import torch

# Set working directory and Python path
os.chdir('/content/VisionDocPhi-3.5')

# CRITICAL: Clear cached modules to ensure fresh imports
for mod in list(sys.modules.keys()):
    if 'src' in mod or 'config' in mod:
        del sys.modules[mod]

sys.path.insert(0, '/content/VisionDocPhi-3.5')

print("Importing project modules...\n")

try:
    # Import configuration
    from config.settings import (
        PROJECT_ROOT, MODEL_NAME, DEVICE, 
        VAL_ANNOTATIONS, IMAGES_DIR, BATCH_SIZE, NUM_WORKERS
    )
    print("  ✓ Configuration loaded")

    # Import models
    from src.models.inference import DocVQAInference
    print("  ✓ Model inference class loaded")

    # Import data utilities
    from src.data.dataset import create_dataloader, get_dataset_stats
    print("  ✓ Data utilities loaded")

    # Import metrics
    from src.utils.metrics import calculate_metrics, anls_score
    print("  ✓ Metrics utilities loaded")

    # Import pipeline
    from src.pipelines.baseline import run_zero_shot_baseline
    print("  ✓ Pipeline utilities loaded")

    print("\n✅ All modules imported successfully!")
    
except Exception as e:
    print(f"\n❌ Import error: {e}")
    print("\nDebugging info:")
    print(f"  Current directory: {os.getcwd()}")
    print(f"  Python path: {sys.path[:2]}")
    import traceback
    traceback.print_exc()

Importing project modules...



ModuleNotFoundError: No module named 'config'

## Step 6: Display Configuration

In [ ]:
import os
import sys
import torch

# Ensure proper setup
os.chdir('/content/VisionDocPhi-3.5')
sys.path.insert(0, '/content/VisionDocPhi-3.5')

# Re-import in case of kernel issues
from config.settings import (
    PROJECT_ROOT, MODEL_NAME, DEVICE, 
    VAL_ANNOTATIONS, IMAGES_DIR, BATCH_SIZE
)

print("\n" + "="*70)
print("⚙️  PROJECT CONFIGURATION")
print("="*70)

current_device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n📍 Model: {MODEL_NAME}")
print(f"💾 Device: {current_device}")
print(f"📂 Project Root: {PROJECT_ROOT}")
print(f"🖼️  Images: {IMAGES_DIR}")
print(f"📋 Annotations: {VAL_ANNOTATIONS}")
print(f"⚡ Batch Size: {BATCH_SIZE}")

print("\n" + "="*70)


⚙️  PROJECT CONFIGURATION


NameError: name 'MODEL_NAME' is not defined

## Step 7: Get Dataset Statistics

In [ ]:
import os
import sys

# Setup
os.chdir('/content/VisionDocPhi-3.5')
sys.path.insert(0, '/content/VisionDocPhi-3.5')

from config.settings import VAL_ANNOTATIONS
from src.data.dataset import get_dataset_stats

print("📊 Dataset Statistics:\n")

try:
    stats = get_dataset_stats(str(VAL_ANNOTATIONS))

    print(f"  Total Samples: {stats['total_samples']}")
    print(f"  Question Types: {stats['num_question_types']}")

    if stats['question_types']:
        print(f"\n  Breakdown by Type:")
        for qtype, count in sorted(stats['question_types'].items(), key=lambda x: x[1], reverse=True):
            pct = (count / stats['total_samples']) * 100
            print(f"    - {qtype}: {count} ({pct:.1f}%)")

    print()
except Exception as e:
    print(f"⚠️  Could not load dataset stats: {e}")
    print("Make sure you've downloaded the images and annotations!")

## Step 8: Initialize Model

In [ ]:
import os
import sys
import torch

# Setup
os.chdir('/content/VisionDocPhi-3.5')

# Clear old modules
for mod in list(sys.modules.keys()):
    if 'src' in mod:
        del sys.modules[mod]

sys.path.insert(0, '/content/VisionDocPhi-3.5')

from config.settings import MODEL_NAME, USE_8BIT_QUANTIZATION
from src.models.inference import DocVQAInference

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("🚀 Initializing Phi-3.5 Vision model...\n")
print(f"Device: {device}")
print(f"Model: {MODEL_NAME}")
print(f"Quantization: {'8-bit (via bitsandbytes)' if USE_8BIT_QUANTIZATION and device == 'cuda' else 'Full precision'}\n")

inference = DocVQAInference(model_name=MODEL_NAME, device=device)

# Verify model has required methods
if not hasattr(inference.model, 'generate'):
    raise RuntimeError("❌ Model loaded but missing 'generate' method! This indicates a loading issue.")

print("✅ Model validation successful - ready for inference!")


## Step 9: Quick Test (5 Samples)

In [ ]:
import json
import os
import sys
from PIL import Image
from tqdm import tqdm

# Setup
os.chdir('/content/VisionDocPhi-3.5')

# Clear old modules if needed
for mod in list(sys.modules.keys()):
    if 'src' in mod:
        del sys.modules[mod]

sys.path.insert(0, '/content/VisionDocPhi-3.5')

from config.settings import VAL_ANNOTATIONS, IMAGES_DIR
from src.data.dataset import create_dataloader
from src.utils.metrics import calculate_metrics, anls_score

print("\n" + "="*70)
print("🧪 QUICK TEST (First 5 Samples) + METRICS")
print("="*70 + "\n")

try:
    # Create dataloader
    dataloader = create_dataloader(
        annotations_file=str(VAL_ANNOTATIONS),
        image_dir=str(IMAGES_DIR),
        split='val',
        batch_size=1,
        num_workers=0,
        shuffle=False
    )

    print(f"✓ Loaded {len(dataloader)} samples from 'val' split\n")

    test_results = []
    predictions = []
    ground_truths = []

    for i, batch in enumerate(dataloader):
        if i >= 5:  # Only test 5 samples
            break
        
        for sample in batch:
            image = sample['image']
            question = sample['question']
            ground_truth = sample['answers'][0] if sample['answers'] else "N/A"
            question_id = sample['question_id']
            
            # Generate answer
            predicted_answer = inference.generate_answer(image, question)
            
            print(f"Sample {i+1}:")
            print(f"  Q: {question}")
            print(f"  Predicted: {predicted_answer}")
            print(f"  Ground Truth: {ground_truth}")
            print()
            
            test_results.append({
                'question': question,
                'predicted': predicted_answer,
                'ground_truth': ground_truth,
                'question_id': question_id
            })
            
            # Store for metrics calculation
            predictions.append(predicted_answer)
            ground_truths.append([ground_truth])  # metrics expect list of answers

    print("\n" + "="*70)
    print("📊 METRICS ON 5 SAMPLES")
    print("="*70 + "\n")
    
    # Calculate metrics
    try:
        metrics = calculate_metrics(predictions, ground_truths)
        
        print("Results:")
        print(f"  ANLS Score: {metrics.get('anls', 0):.4f}")
        print(f"  Exact Match: {metrics.get('exact_match', 0):.4f}")
        
        if 'total_samples' in metrics:
            print(f"  Samples Evaluated: {metrics['total_samples']}")
        
        print("\n✅ Quick test & metrics completed!")
        
    except Exception as e:
        print(f"⚠️  Could not calculate metrics: {e}")
        print("But predictions are ready above!")
        
except Exception as e:
    print(f"❌ Error during quick test: {e}")
    import traceback
    traceback.print_exc()

## Step 10: Full Evaluation Pipeline

**Note:** This runs the complete evaluation. Depending on dataset size, it may take significant time.

In [ ]:
import os
import sys

# Setup
os.chdir('/content/VisionDocPhi-3.5')
sys.path.insert(0, '/content/VisionDocPhi-3.5')

from src.pipelines.baseline import run_zero_shot_baseline

print("\n" + "="*70)
print("📊 FULL EVALUATION PIPELINE")
print("="*70 + "\n")

try:
    # Run using the production pipeline
    eval_results = run_zero_shot_baseline(
        split="val",
        num_samples=None,  # Use all samples
        save_results=True
    )

    print("\n✅ Evaluation completed!")
except Exception as e:
    print(f"❌ Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

## Step 11: Display Results

In [ ]:
import os

print("\n" + "="*70)
print("📈 FINAL RESULTS")
print("="*70 + "\n")

try:
    # Extract metrics
    metrics = {k: v for k, v in eval_results.items() if k != 'results'}

    print("Metrics:")
    for metric_name, metric_value in metrics.items():
        if isinstance(metric_value, float):
            print(f"  {metric_name}: {metric_value:.4f}")
        else:
            print(f"  {metric_name}: {metric_value}")

    print(f"\nResults saved to: data/outputs/")
    print(f"  - predictions_zeroshot.json (detailed predictions)")
    print(f"  - results_zeroshot.json (metrics summary)")

    print("\n" + "="*70)
except Exception as e:
    print(f"⚠️  Could not display results: {e}")
    print("Make sure you've run the full evaluation first!")

## Step 12: Download Results (Optional)

In [ ]:
from google.colab import files
import os

print("\n📥 Downloading results...\n")

# Download results if they exist
results_dir = 'data/outputs'
if os.path.exists(results_dir):
    for file in os.listdir(results_dir):
        if file.endswith('.json'):
            file_path = os.path.join(results_dir, file)
            files.download(file_path)
            print(f"✓ {file}")
    print("\n✅ Download complete!")
else:
    print("⚠️  No results directory found. Run evaluation first!")

## Quantization & Memory Optimization

### 8-Bit Quantization (Default)

The model uses **8-bit quantization via bitsandbytes**, which:
- Reduces model memory footprint by ~75%
- Maintains model accuracy (minimal impact)
- Enables inference on GPU with limited VRAM (e.g., T4 with 16GB)
- Automatically applied when running on CUDA devices

**Requirements:**
```bash
pip install bitsandbytes>=0.43.0
```

**To disable quantization** (if needed), edit [config/settings.py](config/settings.py):
```python
USE_8BIT_QUANTIZATION = False  # Set to False to use full precision
```

### Memory Optimization Features

All enabled by default:
- **Gradient Checkpointing**: Saves memory during inference
- **Eager Attention**: Compatible with Colab, optimized for memory
- **Low CPU Memory Mode**: Efficient model loading
- **Automatic Cache Cleanup**: Every 5 batches

### GPU Memory Usage Estimate

| Mode | GPU Memory | Notes |
|------|-----------|-------|
| **With 8-bit Quantization** | ~10-12 GB | Recommended for Colab T4 |
| **Full Precision (float16)** | ~28-32 GB | Requires larger GPU |
| **Full Precision (float32)** | ~35-40 GB | Not recommended |

---

## Troubleshooting

### Common Issues:

1. **"FileNotFoundError" for data files**
   - Download dataset: https://rrc.cvc.uab.es/?ch=17
   - Extract to: `data/raw/spdocvqa_images/` and `data/raw/spdocvqa_qas/`

2. **"ModuleNotFoundError" when importing**
   - Ensure you're in the correct project directory
   - Check that all `__init__.py` files exist in src/ subdirectories

3. **"ImportError: cannot import name 'BitsAndBytesConfig'"**
   - Run: `pip install --upgrade bitsandbytes transformers`
   - Ensure bitsandbytes version >= 0.43.0

4. **Out of Memory (OOM) Error**
   - Runtime → Change runtime type → Select GPU T4
   - Check Step 3 (Install Dependencies) - ensure bitsandbytes is installed
   - Reduce batch size (already set to 1)
   - Use chunked evaluation: `python scripts/chunked_evaluation.py`

5. **Model Download Fails**
   - Check internet connection
   - Try again (HuggingFace can be slow)
   - Alternative: Download model manually and provide local path

### To Use Different Dataset Split:

```python
# Run on test set
eval_results = run_zero_shot_baseline(split="test")
```

### To Evaluate on Subset:

```python
# Evaluate only first 100 samples
eval_results = run_zero_shot_baseline(split="val", num_samples=100)
```

### To Disable Quantization (Advanced):

If you need full precision for some reason:

1. Edit [config/settings.py](config/settings.py):
```python
USE_8BIT_QUANTIZATION = False
```

2. Restart the runtime and reinitialize the model

**Warning**: This will require significantly more GPU memory (~28-32 GB for float16)
